# PYTORCH RNN trên retailrocket

Nguyễn Ngọc Hoàng Nam — B23DCCN585 | ASG 06

Các cell nhỏ được sắp theo thứ tự phụ thuộc. Huấn luyện đầy đủ mặc định tắt; số liệu chạy thử không phải kết quả test chính thức.

In [1]:
from pathlib import Path
import sys, json, os
ROOT=Path.cwd().resolve()
if ROOT.name=="notebooks": ROOT=ROOT.parent
assert (ROOT/"src").is_dir()
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from src.paths import DATA
from src.io import read_json
import numpy as np
import pandas as pd
from IPython.display import display, Image
print("Project:", ROOT)
print("Data:", DATA)


Project: E:\PTHTTM\ASG_06
Data: E:\PTHTTM\ASG_06_data


## 1. Dữ liệu và hình dạng

In [2]:
from src.data import load_data,split_ids,make_batch
data=load_data('retailrocket')
x,lengths,y=make_batch(data,split_ids(data,'train')[:8])
print(x.shape,lengths,y)

(8, 20, 10) [1 2 3 4 5 6 7 1] [0 0 0 0 0 0 0 0]


## 2. Hàm xây dựng RNN cơ bản

In [3]:
from src.backends import pytorch_runtime
torch,device=pytorch_runtime(42,cpu=True)
print(torch.__version__,device)

2.5.1+cu121 cpu


In [4]:
def build_pytorch(torch, input_dim, hidden, output_dim):
    class SimpleRNN(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn=torch.nn.RNN(input_dim,hidden,batch_first=True,nonlinearity="tanh")
            self.rnn.bias_hh_l0.requires_grad_(False)
            self.head=torch.nn.Linear(hidden,output_dim)
        def forward(self,x,lengths):
            packed=torch.nn.utils.rnn.pack_padded_sequence(
                x,lengths.cpu(),batch_first=True,enforce_sorted=False)
            _,state=self.rnn(packed)
            return self.head(state[-1])
    return SimpleRNN()

In [5]:
model=build_pytorch(torch,data['metadata']['input_dim'],32,3)
print(model)

SimpleRNN(
  (rnn): RNN(10, 32, batch_first=True)
  (head): Linear(in_features=32, out_features=3, bias=True)
)


## 3. Lan truyền thuận và hàm mất mát

In [6]:
tensor=torch.tensor(x)
output=model(tensor,torch.tensor(lengths))
print('Output:',output.shape)

Output: torch.Size([8, 3])


In [7]:
loss=torch.nn.functional.cross_entropy(output,torch.tensor(y,dtype=torch.long))
loss.backward()
print('Loss minh họa:',float(loss))
print('Norm gradient:',float(model.rnn.weight_ih_l0.grad.norm()))

Loss minh họa: 1.0431675910949707
Norm gradient: 0.8006506562232971


## 4. Bộ huấn luyện đầy đủ
Ví dụ trên giải thích đồ thị. Bộ huấn luyện dùng cùng họ kiến trúc, khởi tạo chung giữa framework, trọng số lớp cho Retailrocket, clipping và checkpoint. Mã thuật toán dùng cho lượt chính nằm trong src/backends.py và src/training.py.

In [8]:
from src.training import configuration
config,_=configuration('retailrocket','pytorch')
display(config)

{'seed': 42,
 'hidden_size': 32,
 'batch_size': 256,
 'max_epochs': 20,
 'patience': 4,
 'min_delta': 1e-05,
 'learning_rate': 0.001,
 'adam_epsilon': 1e-07,
 'clip_norm': 1,
 'dataset': 'retailrocket',
 'framework': 'pytorch',
 'smoke': False,
 'class_weights': [0.778438758214022, 3.0554776934025285, 5.659938637800053],
 'metadata_sha256': '08e6797558192e232e85cf83c11f7dea1b86ea1593586222fbcc27a4b0845620',
 'selection': 'validation macro-F1'}

## 5. Chủ động chạy cấu hình này
Chỉ chuyển RUN_FULL=True khi muốn huấn luyện đầy đủ. Nên dùng RUN_TRAINING.bat để giảm bộ nhớ của kernel minh họa đang mở.

In [9]:
RUN_FULL=False
if RUN_FULL:
    import subprocess
    environment=os.environ.copy()
    environment.pop('CUDA_VISIBLE_DEVICES',None)
    subprocess.run([sys.executable,'-m','src.training','--dataset','retailrocket','--framework','pytorch'],cwd=ROOT,env=environment,check=True)
else:
    print('Cell này không khởi động huấn luyện; kết quả đã lưu được đọc ở phần sau.')

Cell này không khởi động huấn luyện; kết quả đã lưu được đọc ở phần sau.


## 6. Dự đoán và kết quả được lưu

In [10]:
folder=ROOT/'results/runs/retailrocket_pytorch_seed42'
if (folder/'metrics.json').exists():
    display(read_json(folder/'metrics.json'))
    display(pd.read_csv(folder/'predictions.csv').head(12))
else:
    print('Kết quả chính sẽ có sau khi bạn huấn luyện xong.')

{'accuracy': 0.9009239204007039,
 'macro_f1': 0.5821070354259991,
 'log_loss': 0.3406922906541197,
 'per_class': [{'label': 0,
   'precision': 0.9478454913460032,
   'recall': 0.9499126706157415,
   'f1': 0.9488779551182047,
   'support': 108211},
  {'label': 1,
   'precision': 0.39285714285714285,
   'recall': 0.27346088767408566,
   'f1': 0.3224618218095311,
   'support': 7683},
  {'label': 2,
   'precision': 0.36161018876506706,
   'recall': 0.6919060052219321,
   'f1': 0.4749813293502614,
   'support': 2298}],
 'ap_1': 0.27549700789783715,
 'ap_2': 0.460469016552754,
 'confusion_matrix': [[102791, 2947, 2473],
  [5248, 2101, 334],
  [408, 300, 1590]],
 'dataset': 'retailrocket',
 'framework': 'pytorch',
 'seed': 42,
 'best_epoch': 8,
 'completed_epochs': 12,
 'train_seconds': 500.01522070000647,
 'samples': 118192,
 'trainable_parameters': 1475,
 'smoke': False,
 'evaluation_split': 'test'}

,sample_id,timestamp,group,actual,prediction
0,0,1442004759,0,0,0
1,1,1442004917,0,0,0
2,9,1440916823,6,0,0
3,10,1440917649,6,0,0
4,11,1440917793,6,0,0
5,20,1441495318,51,0,0
6,21,1441495416,51,0,0
7,22,1441495582,51,0,0
8,23,1441495634,51,0,0
9,24,1441495684,51,0,0
